# EDA: Calima vs Deaths — La Palma

**Objective:** Analyze the association between calima (proxy) and weekly mortality in La Palma, including lagged effects (lag0, lag1, lag2).

**Key variables:**
- `deaths_week`: weekly deaths (2016–2025)
- `calima_proxy_score`: heuristic index (0–4.5)
- `calima_proxy_level`: category (no_calima / possible / probable / intense)

**Sections:**
1. Load data
2. Lag0, lag1, lag2 correlations
3. Group by proxy category
4. Significant differences (ANOVA) and effect sizes (Δ deaths)
4.1 Pairwise comparisons
5. Visualizations
6. Summary

---

## 1. Load Data

In [6]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# ─── ISLAND CONFIG ─────────────────────────────────────────────────────────────
ISLAND_NAME = "la_palma"   # e.g. "gran_canaria", "tenerife", "lanzarote"
ISLAND_CODE = "lpa"   # e.g. "gcan", "tfe", "lanz"
# ───────────────────────────────────────────────────────────────────────────────

CWD = Path.cwd().resolve()

# If running from islands/<island>, go up two levels to repo root
if CWD.name == ISLAND_NAME and CWD.parent.name == "islands":
    ROOT = CWD.parent.parent
else:
    ROOT = CWD

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("CWD :", CWD)
print("ROOT:", ROOT)
print("src exists?:", (ROOT / "src").exists())

from src.utils.d25_nb_utils import (
    section, glance, checks, missing_table, num_summary,
    autosave_fig, save_table,
)

# Output directories
REPORTS_DIR = ROOT / "reports" / "islands"
FIG_DIR = REPORTS_DIR / "figures" / ISLAND_NAME
TAB_DIR = REPORTS_DIR / "tables" / ISLAND_NAME

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print("FIG_DIR:", FIG_DIR)
print("TAB_DIR:", TAB_DIR)

# Load master dataset
FP = ROOT / "data/processed" / ISLAND_NAME / "master" / f"master_{ISLAND_CODE}_2016_2025.parquet"
print("FP:", FP)
assert FP.exists(), f"Missing file: {FP}"

section(f"EDA core weekly {ISLAND_NAME}")

df = pd.read_parquet(FP)
print("Loaded:", FP)

df["week_start"] = pd.to_datetime(df["week_start"], errors="coerce")
print("Week range:", df["week_start"].min(), "->", df["week_start"].max())
glance(df, label=f"eda_core_weekly_{ISLAND_CODE}", n=5)

checks(
    df,
    required=["week_start", "deaths_week"],
    key=["week_start"],
    dt="week_start"
)

num_summary(df)

# Load calima proxy dataset
calima_fp = ROOT / "data" / "processed" / ISLAND_NAME / "calima" / f"calima_proxy_v2_weekly_{ISLAND_CODE}_2016_2025.parquet"
print("Calima proxy FP:", calima_fp)

if calima_fp.exists():
    calima = pd.read_parquet(calima_fp).copy()
    calima["week_start"] = pd.to_datetime(calima["week_start"], errors="coerce")
    keep = [
        "week_start",
        "calima_proxy_score",
        "calima_proxy_level",
    ]

    calima_keep = [c for c in keep if c in calima.columns]

    # Drop overlapping columns before merge to avoid duplicates
    overlap = [c for c in calima_keep if c != "week_start" and c in df.columns]
    if overlap:
        print("Dropping overlapping columns before merge:", overlap)
        df = df.drop(columns=overlap)

    df = df.merge(calima[calima_keep], on="week_start", how="left")
    print("Merged calima proxy columns:", [c for c in calima_keep if c != "week_start"])
else:
    print("Calima proxy weekly dataset not found. Section 6.1 will be skipped.")

print("Final shape:", df.shape)

CWD : C:\Users\fdora\RA_Career\Projects\climate_mortality\islands\la_palma
ROOT: C:\Users\fdora\RA_Career\Projects\climate_mortality
src exists?: True
FIG_DIR: C:\Users\fdora\RA_Career\Projects\climate_mortality\reports\islands\figures\la_palma
TAB_DIR: C:\Users\fdora\RA_Career\Projects\climate_mortality\reports\islands\tables\la_palma
FP: C:\Users\fdora\RA_Career\Projects\climate_mortality\data\processed\la_palma\master\master_lpa_2016_2025.parquet

EDA core weekly la_palma
Loaded: C:\Users\fdora\RA_Career\Projects\climate_mortality\data\processed\la_palma\master\master_lpa_2016_2025.parquet
Week range: 2015-12-28 00:00:00 -> 2025-12-29 00:00:00

--- eda_core_weekly_lpa ---
shape: (523, 42)

dtypes:
 week_start                     datetime64[ns]
year                                    int32
island                                 object
island_code                            object
deaths_week                           float64
deaths_missing_week                   float64
n_days       

,week_start,year,island,island_code,deaths_week,deaths_missing_week,n_days,temp_c_mean,tmax_c_mean,tmax_c_max,...,O3,days_with_pm10,days_missing_pm10,cap_heat_level_max_week,cap_dust_level_max_week,cap_heat_yellow_plus_week,cap_dust_yellow_plus_week,cap_coverage_week,calima_dai_flag,calima_level_week
0,2015-12-28,2015,la_palma,lpa,NaN,NaN,3,18.933333,21.966667,22.2,...,71.333333,3,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
1,2016-01-04,2016,la_palma,lpa,17.0,0.0,7,19.314286,21.957143,23.9,...,77.000000,7,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
2,2016-01-11,2016,la_palma,lpa,22.0,0.0,7,20.371429,23.085714,23.6,...,68.428571,7,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
3,2016-01-18,2016,la_palma,lpa,12.0,0.0,7,19.828571,22.985714,25.4,...,70.571429,7,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0
4,2016-01-25,2016,la_palma,lpa,13.0,0.0,7,19.800000,22.371429,23.8,...,82.571429,7,0,NaN,NaN,NaN,NaN,NaN,0.0,0.0


Calima proxy FP: C:\Users\fdora\RA_Career\Projects\climate_mortality\data\processed\la_palma\calima\calima_proxy_v2_weekly_lpa_2016_2025.parquet
Merged calima proxy columns: ['calima_proxy_score', 'calima_proxy_level']
Final shape: (523, 44)


## 2. Lags

In [7]:
# Filter out the first partial week (null deaths)
first_week = df['week_start'].min()
df = df[df['week_start'] > first_week].reset_index(drop=True)

print(f"Rows after filtering first week: {len(df)}")
print(f"Deaths nulls: {df['deaths_week'].isna().sum()}")
print(f"Calima proxy score nulls: {df['calima_proxy_score'].isna().sum()}")

# Create lag variables for calima_proxy_score
df['calima_proxy_score_lag1'] = df['calima_proxy_score'].shift(1)
df['calima_proxy_score_lag2'] = df['calima_proxy_score'].shift(2)

print("\n✅ Lag variables created:")
print(f"  lag0 (contemporaneous): {df['calima_proxy_score'].notna().sum()} non-null")
print(f"  lag1 (1 week prior):    {df['calima_proxy_score_lag1'].notna().sum()} non-null")
print(f"  lag2 (2 weeks prior):   {df['calima_proxy_score_lag2'].notna().sum()} non-null")

Rows after filtering first week: 522
Deaths nulls: 0
Calima proxy score nulls: 0

✅ Lag variables created:
  lag0 (contemporaneous): 522 non-null
  lag1 (1 week prior):    521 non-null
  lag2 (2 weeks prior):   520 non-null


In [8]:
# Correlations: deaths_week vs calima_proxy_score at different lags
corr_lag0 = df['deaths_week'].corr(df['calima_proxy_score'])
corr_lag1 = df['deaths_week'].corr(df['calima_proxy_score_lag1'])
corr_lag2 = df['deaths_week'].corr(df['calima_proxy_score_lag2'])

corr_summary = pd.DataFrame({
    'lag': ['lag0 (same week)', 'lag1 (1 week prior)', 'lag2 (2 weeks prior)'],
    'correlation': [corr_lag0, corr_lag1, corr_lag2],
    'n_pairs': [
        df[['deaths_week', 'calima_proxy_score']].notna().all(axis=1).sum(),
        df[['deaths_week', 'calima_proxy_score_lag1']].notna().all(axis=1).sum(),
        df[['deaths_week', 'calima_proxy_score_lag2']].notna().all(axis=1).sum(),
    ]
})

print("Correlations: deaths_week vs calima_proxy_score\n")
print(corr_summary.to_string(index=False))

# Save
corr_summary.to_csv(TAB_DIR / 'calima_deaths_lags_correlation.csv', index=False)
print("\n✅ Saved: calima_deaths_lags_correlation.csv")

Correlations: deaths_week vs calima_proxy_score

                 lag  correlation  n_pairs
    lag0 (same week)     0.019906      522
 lag1 (1 week prior)    -0.102697      521
lag2 (2 weeks prior)    -0.037150      520

✅ Saved: calima_deaths_lags_correlation.csv


**Lag Analysis — La Palma:**

| Lag | Correlation | Interpretation |
|-----|-------------|-----------------|
| **lag0 (same week)** | 0.020 | **Strongest (but very weak)** ✓ |
| lag1 (1 week prior) | -0.103 | **Negative** ⚠️ |
| lag2 (2 weeks prior) | -0.037 | Weak negative |

**Interpretation:**

✓ **Strongest at lag0 (r=0.020)** → Technically strongest, but **essentially zero correlation**.

**Key findings:**

1. **No meaningful correlation:**
   - lag0 = 0.020 is essentially **noise** (r near 0)
   - Comparison: Tenerife 0.221, Gran Canaria 0.210, Lanzarote 0.120
   - La Palma is **~10x weaker** than smallest island (Lanzarote)

2. **Negative lags (unusual pattern):**
   - lag1 and lag2 are **negative** (-0.103, -0.037)
   - Suggests calima 1-2 weeks PRIOR may be associated with **lower** mortality
   - Opposite of what we'd expect → likely statistical artifact/noise

3. **Population confounding:**
   - La Palma: ~17k deaths/year (~0.33 deaths/day)
   - Extreme noise-to-signal ratio
   - Weekly variability dominates any calima effect

**Comparison across islands:**

| Island | lag0 | Pattern | n population |
|--------|------|---------|---|
| Tenerife | 0.221 | Strong positive | 928k |
| Gran Canaria | 0.210 | Strong positive | 846k |
| Lanzarote | 0.120 | Weak positive | 149k |
| **La Palma** | **0.020** | **No signal** ⚠️ | **83k** |

**Epidemiological implication:**
- La Palma too small to detect calima-mortality correlation reliably
- Weekly deaths too variable (high noise floor)
- Effect may exist but **undetectable** in this population size

**Conclusion:** La Palma shows **no meaningful correlation** between calima and mortality. Population size too small for reliable lag analysis.

---

## 3. Group by Proxy Category

In [9]:

# Group by calima_proxy_level and compute deaths statistics
level_order = ['no_calima', 'possible', 'probable', 'intense']

deaths_by_level = (
    df.groupby('calima_proxy_level', observed=True)['deaths_week']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .reindex(level_order)
)

print("Deaths statistics by calima proxy level:\n")
print(deaths_by_level.round(2))

# Compute Δ deaths (intense vs baseline)
baseline = deaths_by_level.loc['no_calima', 'mean']
intense  = deaths_by_level.loc['intense', 'mean']
delta    = intense - baseline

print(f"\nΔ deaths (intense vs no_calima): {delta:.2f} deaths/week")

# Save
deaths_by_level.to_csv(TAB_DIR / 'calima_level_v_deaths_stats.csv')
print("\n✅ Saved: calima_level_v_deaths_stats.csv")

Deaths statistics by calima proxy level:

                    count   mean  median   std  min   max
calima_proxy_level                                       
no_calima             378  15.75    15.0  4.77  5.0  48.0
possible               71  16.08    16.0  4.25  9.0  28.0
probable               50  15.42    15.0  4.38  7.0  31.0
intense                23  16.87    15.0  6.11  7.0  31.0

Δ deaths (intense vs no_calima): 1.12 deaths/week

✅ Saved: calima_level_v_deaths_stats.csv


**Interpretation: Deaths by Calima Proxy Level — La Palma:**

**Finding:** No clear dose-response; marginal effect at intense only.

| Level | Mean deaths/week | Δ vs baseline | % increase |
|-------|-----------------|---------------|-----------|
| **no_calima** | 15.75 | — | — |
| **possible** | 16.08 | +0.33 | +2.1% |
| **probable** | 15.42 | -0.33 | -2.1% |
| **intense** | 16.87 | **+1.12** | **+7.1%** |

**Key findings:**

1. **No monotonic gradient:**
   - no_calima → possible: +2.1%
   - no_calima → probable: -2.1% (INVERSE)
   - no_calima → intense: +7.1%
   - Pattern is **erratic, not dose-dependent**

2. **Marginal intense effect:**
   - Only intense shows meaningful excess (+1.12 deaths/week)
   - Δ much smaller than other islands (Tenerife 17.19, Gran Canaria 17.97, Lanzarote 2.16)
   - **Smallest effect seen so far**

3. **High variability & small sample:**
   - intense weeks: n=23 (smallest sample)
   - std increases at intense (6.11), indicating high noise
   - Suggests effect is **fragile/unstable**

4. **Comparison with other islands:**

| Island | n_intense | Δ deaths | % increase |
|--------|-----------|----------|-----------|
| Tenerife | 49 | 17.19 | 12.4% |
| Gran Canaria | 46 | 17.97 | 13.5% |
| Lanzarote | 41 | 2.16 | 13.7% |
| **La Palma** | **23** | **1.12** | **7.1%** |

**Epidemiological interpretation:**
- La Palma's smallest population (~85k) → **highest noise-to-signal ratio**
- Probable category shows *negative* excess → strongly suggests **random noise**
- Intense effect (+7.1%) smallest of all islands
- Effect size proportional to **population size** not **calima exposure**

**Conclusion:** La Palma shows **weak, unstable pattern** with erratic responses across calima levels. No clear dose-response. Effect magnitude marginal and likely confounded by high weekly variability.

---

## 4. Significant Differences (ANOVA) and Effect Sizes (Δ deaths)

In [10]:
from scipy import stats

# ANOVA: Are there significant differences across groups?
groups = [df[df['calima_proxy_level'] == level]['deaths_week'].dropna()
          for level in level_order]
f_stat, p_value = stats.f_oneway(*groups)

print("ANOVA: Deaths across calima proxy levels")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value:     {p_value:.6f}")

if p_value < 0.05:
    print("✅ Significant difference (p < 0.05)")
else:
    print("⚠️ Not significant at α=0.05")

# Effect size: eta-squared (η²)
grand_mean = df['deaths_week'].mean()
ss_between = sum(len(groups[i]) * (groups[i].mean() - grand_mean)**2
                 for i in range(len(groups)))
ss_total   = sum((df['deaths_week'] - grand_mean)**2)
eta_squared = ss_between / ss_total

print(f"\nEffect size (η²): {eta_squared:.4f}")
print(f"  0.01 = small | 0.06 = medium | 0.14+ = large")

ANOVA: Deaths across calima proxy levels
F-statistic: 0.5975
P-value:     0.616854
⚠️ Not significant at α=0.05

Effect size (η²): 0.0034
  0.01 = small | 0.06 = medium | 0.14+ = large


**Interpretation: ANOVA & Effect Size — La Palma:**

| Metric | Value | Interpretation |
|--------|-------|-----------------|
| **F-statistic** | 0.60 | Very weak evidence |
| **P-value** | 0.617 | **NOT significant (p >> 0.05)** ✗ |
| **η² (eta-squared)** | 0.0034 | **Negligible effect size** |

**Key insights:**

1. **No statistical significance:**
   - p = 0.617 (far above 0.05 threshold)
   - Cannot reject null hypothesis: mortality differences across calima levels are **due to chance**
   - First island with **non-significant ANOVA**

2. **Negligible effect size:**
   - η² = 0.0034 → calima explains only **0.34% of mortality variance**
   - Classification: **Far below small (0.01)**
   - Essentially no explanatory power

**Comparison across all islands:**

| Island | F-stat | p-value | η² | Significance |
|--------|--------|---------|-----|--------------|
| Tenerife | 9.87 | 2.4e-06 | 0.054 | **✓✓ Strong** |
| Gran Canaria | 10.55 | 9.6e-07 | 0.058 | **✓✓ Strong** |
| Lanzarote | 2.73 | 4.3e-02 | 0.016 | **✓ Weak** |
| **La Palma** | **0.60** | **0.617** | **0.0034** | **✗ None** |

**Epidemiological implication:**
- La Palma population **too small** to detect calima-mortality signal
- Weekly mortality noise >> calima effect size
- Statistical power insufficient for reliable effect estimation

**Critical finding:**
- Calima proxy v2 works for islands ≥140k (Lanzarote)
- Below that threshold, effect is **undetectable**

**Conclusion:** La Palma shows **no statistically significant calima-mortality association**. Population size insufficient to isolate effect from background noise. v2 proxy validity limited to larger islands.

---

**La Palma: ANOVA non-significant (p=0.617, η²=0.0034). Population size insufficient to detect calima-mortality association. Analysis discontinued.**